# Colab 06 — ¿Cambia k si tomo en serio las barras de error?

**Laboratorio 1 · Departamento de Física · FCEN-UBA**

Clase 6 — 16/09

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/main/06_Ajuste_ponderado_covarianza_y_chi2.ipynb)

Hoy volviste a medir el resorte, pero repitiendo cada punto varias veces, así que las barras de error las **mediste** en lugar de estimarlas. Y medistes dos resortes en serie y en paralelo, que es una predicción para poner a prueba.

**Al terminar vas a poder:** ajustar con pesos, obtener incertezas confiables de los parámetros, y decidir con $\chi^2_\nu$ y p-valor si tu modelo describe los datos.

---

### Antes de tocar nada

Andá a **Archivo → Guardar una copia en Drive**. Vas a trabajar sobre tu copia:
lo que escribas acá sin copiar primero no se guarda en ningún lado.

Este cuaderno se recorre **de arriba hacia abajo**. Las celdas no son
independientes: cada una usa lo que definieron las anteriores. Si algo tira
`NameError`, casi siempre es porque salteaste una celda.

In [ ]:
import os

if not os.path.exists("lab1_utils.py"):
    !wget -q -O lab1_utils.py https://raw.githubusercontent.com/charlyacha/Labo1-colabs/main/lab1_utils.py

import numpy as np
import matplotlib.pyplot as plt
import lab1_utils as lab

lab.estilo_lab1()
print("Listo. numpy", np.__version__)

### 1. Las barras de error no se inventan: se miden

En la Clase 5 le pusimos a todos los puntos la misma incerteza, estimada de
la resolución de la regla. Hoy hacemos lo correcto: **cada masa se lee
varias veces**, y la incerteza de ese punto es el error de la media de sus
repeticiones.

El resultado es que las barras salen distintas entre sí, y no por capricho:
a masa alta el sistema queda oscilando más tiempo y leer la posición del
indicador cuesta más. Esa heterogeneidad es información física, y tirarla a
la basura promediándola es perder precisión.

In [ ]:
# Esta celda fabrica datos de ejemplo con 4 repeticiones por punto.
# Reemplazala por la lectura de tu archivo cuando tengas tus datos.
generador = np.random.default_rng(11)
g = 9.797


def medir_resorte(k_real, masas, ruido_base=0.00025, semilla=None):
    # Simula 4 lecturas de la elongación para cada masa. El ruido crece con
    # la carga, que es lo que pasa en la mesada.
    gen = np.random.default_rng(semilla)
    x_real = masas * g / k_real
    ruido = ruido_base * (1 + 10 * masas / masas.max())
    return x_real[:, None] + gen.normal(0, ruido[:, None], size=(len(masas), 4))


masas = np.array([50, 100, 150, 200, 250, 300, 350, 400]) * 1e-3
lecturas = medir_resorte(24.8, masas, semilla=32)

elongacion = lecturas.mean(axis=1)
s_elong = np.std(lecturas, axis=1, ddof=1) / np.sqrt(lecturas.shape[1])

fuerza = masas * g

print("  F (N)     x (m)      sigma_x (m)   sigma relativo")
for F, x, s in zip(fuerza, elongacion, s_elong):
    print(f"{F:7.3f}   {x:8.5f}   {s:9.6f}     {100*s/x:5.2f} %")

### 2. Ponderar: qué cambia y por qué

Cuando los puntos tienen incertezas distintas, el ajuste no ponderado es
sencillamente **el ajuste equivocado**: le está dando la misma influencia a
un punto que medís al 1 % y a uno que medís al 10 %.

Cuadrados mínimos ponderados minimiza

$$\chi^2 = \sum_i \frac{(y_i - f(x_i))^2}{\sigma_i^2}$$

es decir, cada residuo se mide **en unidades de su propia barra de error**.

In [ ]:
def recta(x, a, b):
    return a * x + b


print("SIN pesos")
p_sin, e_sin, _ = lab.ajustar(recta, fuerza, elongacion,
                              nombres=["a (m/N)", "b (m)"])

print()
print("CON pesos")
p_con, e_con, cov_con = lab.ajustar(recta, fuerza, elongacion, yerr=s_elong,
                                    nombres=["a (m/N)", "b (m)"])

In [ ]:
k_sin, sk_sin = 1/p_sin[0], e_sin[0]/p_sin[0]**2
k_con, sk_con = 1/p_con[0], e_con[0]/p_con[0]**2

print("k sin pesos:", lab.formatear(k_sin, sk_sin, "N/m"))
print("k con pesos:", lab.formatear(k_con, sk_con, "N/m"))
print()
print(f"diferencia: {abs(k_sin - k_con):.3f} N/m")
print(f"error del valor ponderado: {sk_con:.3f} N/m")
print(f"la diferencia es {abs(k_sin-k_con)/sk_con:.1f} veces el error")

Ahí está contestada la pregunta del título: el cambio es **varias veces
mayor que la incerteza del resultado**. Ponderar no era un detalle de
prolijidad, era la diferencia entre dos respuestas distintas.

**Una advertencia metodológica sobre la comparación que acabamos de hacer.**
No usamos `lab.compatibilidad` acá, y es a propósito. Ese test supone
mediciones **independientes**, y estos dos números salen de *los mismos
datos* analizados de dos maneras: están fuertemente correlacionados, así que
la fórmula $\sqrt{\sigma_1^2 + \sigma_2^2}$ sobreestima muchísimo la
incerteza de la diferencia y el test siempre daría "compatibles". Comparar
dos análisis del mismo conjunto no es lo mismo que comparar dos mediciones.
Lo correcto acá es lo que hicimos: mirar el cambio contra la incerteza del
resultado.

### 3. De dónde salen los errores de los parámetros

`curve_fit` devuelve `popt` y `pcov`. La segunda es la **matriz de
covarianza** de los parámetros: en la diagonal están las varianzas, y fuera
de la diagonal las covarianzas, que dicen cuánto se compensan entre sí los
parámetros.

$$\sigma_{p_i} = \sqrt{\mathrm{pcov}_{ii}}$$

In [ ]:
print("matriz de covarianza:")
print(cov_con)
print()
print("errores de los parámetros:", np.sqrt(np.diag(cov_con)))
print()
print("matriz de correlación:")
print(lab.matriz_correlacion(cov_con).round(3))

La correlación entre pendiente y ordenada es fuerte y negativa, y tiene un
significado geométrico simple: si subís la ordenada, tenés que bajar la
pendiente para que la recta siga pasando por la nube de puntos. Por eso
reportar los dos parámetros por separado con sus errores es una descripción
**incompleta** de lo que sabés.

### 4. `absolute_sigma`: el default está mal para nosotros

Éste es el punto más importante de la clase y el que menos aparece en los
tutoriales.

Por defecto `curve_fit` usa `absolute_sigma=False`, y con esa opción **la
covarianza que devuelve está reescalada por el $\chi^2_\nu$ del propio
ajuste**:

$$\mathrm{pcov}_{\text{devuelta}} = \mathrm{pcov}_{\text{verdadera}} \times \chi^2_\nu$$

O sea: scipy interpreta tus barras de error como **pesos relativos de escala
desconocida** y ajusta la escala global para que el ajuste "cierre". Las
consecuencias son dos, y la segunda es letal para este curso:

1. Los errores de los parámetros salen inflados o desinflados por un factor
   que depende de qué tan bien ajustó el modelo, no de qué tan bien mediste.
2. **Destruye el diagnóstico de bondad de ajuste por construcción.** Si
   escalás para que $\chi^2_\nu = 1$, ya no podés usar $\chi^2_\nu$ para
   decidir si el modelo sirve: la pregunta se contesta sola.

En un laboratorio de física las barras de error son magnitudes físicas
absolutas, no pesos relativos. Va `absolute_sigma=True`, siempre.

In [ ]:
from scipy.optimize import curve_fit

p_T, cov_T = curve_fit(recta, fuerza, elongacion, sigma=s_elong,
                       absolute_sigma=True)
p_F, cov_F = curve_fit(recta, fuerza, elongacion, sigma=s_elong,
                       absolute_sigma=False)

print("                     pendiente          error de la pendiente")
print(f"absolute_sigma=True   {p_T[0]:.6e}     {np.sqrt(cov_T[0,0]):.3e}")
print(f"absolute_sigma=False  {p_F[0]:.6e}     {np.sqrt(cov_F[0,0]):.3e}")
print()
print(f"cociente entre los errores: {np.sqrt(cov_F[0,0]/cov_T[0,0]):.3f}")

chi2 = np.sum(((elongacion - recta(fuerza, *p_T))/s_elong)**2)
print(f"raíz del χ²_ν del ajuste  : {np.sqrt(chi2/(len(fuerza)-2)):.3f}")
print()
print("Son el mismo número: ese es exactamente el factor de reescaleo.")

**La trampa gemela.** `numpy.polyfit(..., cov=True)` hace lo mismo por
defecto y hace falta `cov='unscaled'` para evitarlo. Como `polyfit` es lo
primero que aparece googleando "ajuste lineal python", conviene saberlo
aunque nosotros usemos `curve_fit`.

### 5. `sigma` no es un peso

Otro error silencioso. En `curve_fit`, el argumento `sigma` es la
**desviación estándar** de cada punto. El peso lo construye la rutina
internamente como $w_i = 1/\sigma_i^2$. Si le pasás `1/err` creyendo que le
estás pasando el peso, el ajuste queda ponderado **al revés**: los puntos
peores dominan. No hay ningún mensaje de error.

In [ ]:
p_bien, _ = curve_fit(recta, fuerza, elongacion, sigma=s_elong,
                      absolute_sigma=True)
p_mal, _ = curve_fit(recta, fuerza, elongacion, sigma=1/s_elong,
                     absolute_sigma=True)

print(f"k con sigma=err    : {1/p_bien[0]:.4f} N/m   <- correcto")
print(f"k con sigma=1/err  : {1/p_mal[0]:.4f} N/m   <- ponderado al revés")

### 6. Chi cuadrado reducido y su p-valor

$$\chi^2 = \sum_i \frac{(y_i - f(x_i))^2}{\sigma_i^2}
\qquad
\nu = N - p
\qquad
\chi^2_\nu = \frac{\chi^2}{\nu}$$

Si el modelo es correcto y las barras están bien estimadas, cada término
aporta en promedio 1, así que $\chi^2_\nu \approx 1$. Las desviaciones se
leen así:

- $\chi^2_\nu \gg 1$: el modelo no describe los datos, **o** las incertezas
  están subestimadas. Las dos causas se distinguen mirando los residuos: si
  tienen estructura, es el modelo; si son ruido puro pero grandes, son las
  barras.
- $\chi^2_\nu \ll 1$: incertezas sobreestimadas. Suena a buena noticia y no
  lo es.

Pero "$\chi^2_\nu \approx 1$" es incompleto sin los grados de libertad.

In [ ]:
from scipy import stats

print("¿Es aceptable un χ²_ν = 1,8?  Depende de ν.")
for nu in [3, 5, 10, 30, 100]:
    p = stats.chi2.sf(1.8*nu, nu)
    veredicto = "aceptable" if p > 0.01 else "descartable"
    print(f"  ν = {nu:3d}  ->  p = {p:.2e}   {veredicto}")

Con $\nu = 5$ un $\chi^2_\nu$ de 1,8 pasa sin problemas; con $\nu = 100$ es
imposible por azar. El **p-valor** es la probabilidad de obtener un $\chi^2$
igual o peor si el modelo fuera correcto, y convierte una regla de pulgar en
un criterio. Una línea de código: `stats.chi2.sf(chi2, nu)`.

Corolario poco enseñado: un p-valor **demasiado alto** (> 0,99) es igual de
sospechoso. Significa incertezas infladas o —en el peor caso— datos
retocados. En la historia de la física hay más de un caso célebre detectado
exactamente así.

In [ ]:
c2r, pval = lab.chi2_reducido(elongacion, recta(fuerza, *p_con), s_elong, 2)

fig, axes = lab.grafico_con_residuos(
    fuerza, elongacion, recta, p_con, yerr=s_elong,
    normalizar_residuos=True,
    xlabel="Fuerza aplicada (N)", ylabel="Elongación (m)",
    titulo=f"Ajuste ponderado — χ²_ν = {c2r:.2f}, p = {pval:.3f}")
plt.show()

Los residuos normalizados por sigma son la mejor versión del panel de abajo:
en esas unidades, "estar lejos" tiene un significado absoluto. Dos tercios
de los puntos deberían caer entre −1 y 1, y prácticamente todos entre −2 y 2.

### 7. Serie y paralelo: una predicción para poner a prueba

Con dos resortes de constantes $k_1$ y $k_2$:

$$k_{\text{serie}} = \left(\frac{1}{k_1} + \frac{1}{k_2}\right)^{-1}
\qquad
k_{\text{paralelo}} = k_1 + k_2$$

Esto no es una fórmula para verificar cualitativamente: es una **predicción
con incerteza**, que se propaga desde $k_1$ y $k_2$ y se compara con la
medición directa. Si el $z$ da 0,4, la ley se verificó. Si da 6, hay algo
que no está en el modelo.

In [ ]:
configuraciones = {
    "resorte 1": 24.8,
    "resorte 2": 41.5,
    "serie": 1/(1/24.8 + 1/41.5),
    "paralelo": 24.8 + 41.5,
}

resultados = {}
for i, (nombre, k_real) in enumerate(configuraciones.items()):
    # A mayor k, menos elongación: hay que colgar más masa para medir bien.
    escala = k_real / 24.8
    m = np.array([50, 100, 150, 200, 250, 300, 350, 400]) * 1e-3 * escala
    lec = medir_resorte(k_real, m, semilla=20 + i)
    x, sx = lec.mean(axis=1), np.std(lec, axis=1, ddof=1)/np.sqrt(4)
    p, e, _ = lab.ajustar(recta, m*g, x, yerr=sx, verbose=False)
    resultados[nombre] = (1/p[0], e[0]/p[0]**2)
    print(f"{nombre:<10} {lab.formatear(*resultados[nombre], 'N/m')}")

In [ ]:
k1, sk1 = resultados["resorte 1"]
k2, sk2 = resultados["resorte 2"]

# Paralelo: k = k1 + k2. Los errores se suman en cuadratura.
kp_pred = k1 + k2
skp_pred = np.sqrt(sk1**2 + sk2**2)

# Serie: k = (1/k1 + 1/k2)^-1. Derivadas parciales:
#   dk/dk1 = (k/k1)^2   y   dk/dk2 = (k/k2)^2
ks_pred = 1/(1/k1 + 1/k2)
sks_pred = np.sqrt(((ks_pred/k1)**2 * sk1)**2 + ((ks_pred/k2)**2 * sk2)**2)

print("PARALELO")
lab.compatibilidad(kp_pred, skp_pred, *resultados["paralelo"],
                   etiquetas=("predicho", "medido"))
print()
print("SERIE")
lab.compatibilidad(ks_pred, sks_pred, *resultados["serie"],
                   etiquetas=("predicho", "medido"))

Notá algo que sale gratis y es muy formativo: la incerteza **relativa** del
paralelo predicho es menor que la de cada resorte por separado, mientras que
la del serie está dominada por el resorte más blando. La estructura de la
propagación te dice dónde vale la pena poner el esfuerzo experimental.

### 8. Optativo: ¿y si la variable independiente también tiene error?

Cuadrados mínimos supone $\sigma_x = 0$, y en Hooke esa hipótesis se viola:
la masa colgada también tiene incerteza. El criterio operativo para decidir
si importa es comparar el efecto de $\sigma_x$ **traducido al eje y** contra
$\sigma_y$:

$$\sigma_x \left|\frac{dy}{dx}\right| \ll \sigma_y
\quad\Rightarrow\quad \text{se puede ignorar}$$

Lo obligatorio es **verificar la hipótesis**, no dominar la solución.

In [ ]:
s_masa = 0.0005
s_fuerza = s_masa * g
pendiente = p_con[0]

efecto = s_fuerza * abs(pendiente)

print(f"σ_x · |dy/dx| = {efecto:.2e} m")
print(f"σ_y típico     = {np.mean(s_elong):.2e} m")
print(f"cociente       = {efecto/np.mean(s_elong):.3f}")
print()
if efecto < 0.3*np.mean(s_elong):
    print("Se puede ignorar con tranquilidad.")
else:
    print("No es despreciable: hay que hacer algo.")

Con estos datos el cociente da alrededor de 0,25: ignorar $\sigma_x$
subestima la barra de error en un 3 %, que es tolerable. Pero conviene saber
cómo se corrige, porque en otros experimentos —cualquiera donde la variable
independiente sea un tiempo cronometrado a mano, por ejemplo— no lo es.

La salida más simple, y suficiente para todo lo que hacemos en este curso,
es construir un **sigma efectivo** sumando en cuadratura la contribución de $\sigma_x$
proyectada sobre el eje $y$:

$$\sigma_y^{\text{ef}} = \sqrt{\sigma_y^2 + \left(\sigma_x\,\frac{dy}{dx}\right)^2}$$

Es una aproximación de primer orden, igual que toda la propagación de la
Clase 4. La alternativa completa es `scipy.odr` (regresión por distancias
ortogonales), que queda como material optativo: lo importante en primer año
es **verificar la hipótesis**, no dominar ODR.

In [ ]:
s_efectivo = np.sqrt(s_elong**2 + (s_fuerza*pendiente)**2)

p_ef, e_ef, _ = lab.ajustar(recta, fuerza, elongacion, yerr=s_efectivo,
                            nombres=["a (m/N)", "b (m)"])

k_ef, sk_ef = 1/p_ef[0], e_ef[0]/p_ef[0]**2
print()
print("k ignorando sigma_x :", lab.formatear(k_con, sk_con, "N/m"))
print("k con sigma efectivo:", lab.formatear(k_ef, sk_ef, "N/m"))
print()
print("El valor central casi no cambia; lo que cambia es la incerteza,")
print("que ahora incluye una fuente que antes estábamos ignorando.")

### 9. Ejercicios

1. Rehacé el análisis con tus cuatro configuraciones. ¿Se verifican las dos
   leyes de composición dentro de las incertezas?
2. ¿Qué pasa con el $\chi^2_\nu$ si repetís cada punto **dos** veces en
   lugar de cuatro? Rehacé el análisis quedándote solo con las dos primeras
   lecturas de cada masa y explicá el cambio.
3. Ajustá el conjunto de la Clase 5 (con los puntos de la zona no lineal)
   ponderando, y mirá el p-valor. Comparalo con lo que concluías mirando
   $R^2$.
4. Multiplicá todas tus barras de error por 2 y rehacé el ajuste con
   `absolute_sigma=True` y con `False`. ¿Cuál de los dos cambia los errores
   de los parámetros, y cuál no? Explicá por qué.

In [ ]:
# Espacio de trabajo para los ejercicios.

### Informe 2 (Clases 4 a 6)

Propagación de incertezas y ajuste lineal ponderado, con $\chi^2_\nu$,
p-valor y gráfico de residuos. La verificación de las leyes de composición
de resortes es el resultado central.